In [1]:
import pandas as pd
from sqlalchemy import create_engine
import re 

DB_USER = 'postgres'
DB_PASSWORD = '0000' 
DB_HOST = 'localhost'
DB_PORT = '5433'     
DB_NAME = 'movie_recsys_db' 

# Создание строки подключения SQLAlchemy
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

print(f"Подключение к БД: {DB_NAME} на {DB_HOST}:{DB_PORT} успешно установлено.")

Подключение к БД: movie_recsys_db на localhost:5433 успешно установлено.


In [2]:
# SQL-запрос для извлечения данных о фильмах, их жанрах и ключевых словах
query = """
SELECT
    m.movie_id,
    m.title,
    m.overview,
    m.poster_path,
    STRING_AGG(DISTINCT g.name, ' ') AS genres_list,    -- Объединяем названия жанров в одну строку
    STRING_AGG(DISTINCT k.name, ' ') AS keywords_list  -- Объединяем ключевые слова в одну строку
FROM
    movies m
LEFT JOIN
    movie_genres mg ON m.movie_id = mg.movie_id
LEFT JOIN
    genres g ON mg.genre_id = g.genre_id
LEFT JOIN
    movie_keywords mk ON m.movie_id = mk.movie_id
LEFT JOIN
    keywords k ON mk.keyword_id = k.keyword_id
GROUP BY
    m.movie_id, m.title, m.overview
ORDER BY
    m.movie_id;
"""

print("Извлечение данных из БД...")
try:
    df_movies_content = pd.read_sql_query(query, engine)
    print(f"Загружено {len(df_movies_content)} фильмов с их жанрами и ключевыми словами.")
    print("\nПервые 5 строк загруженных данных:")
    print(df_movies_content.head())
except Exception as e:
    print(f"Ошибка при извлечении данных: {e}")
    df_movies_content = pd.DataFrame() # Создаем пустой DataFrame в случае ошибки

Извлечение данных из БД...
Загружено 9082 фильмов с их жанрами и ключевыми словами.

Первые 5 строк загруженных данных:
   movie_id                        title  \
0         1                    Toy Story   
1         2                      Jumanji   
2         3             Grumpier Old Men   
3         4            Waiting to Exhale   
4         5  Father of the Bride Part II   

                                            overview  \
0  Led by Woody, Andy's toys live happily in his ...   
1  When siblings Judy and Peter discover an encha...   
2  A family wedding reignites the ancient feud be...   
3  Cheated on, mistreated and stepped on, the wom...   
4  Just when George Banks has recovered from his ...   

                        poster_path               genres_list  \
0  /rhIRbceoE9lR4veEXuwCC2wARtG.jpg   Animation Comedy Family   
1  /vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg  Adventure Family Fantasy   
2  /6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg            Comedy Romance   
3  /16XOMpEaLWkrcP

In [3]:
if not df_movies_content.empty:
    # Очистка и объединение текстовых данных
    
    # Заполняем NaN значения в текстовых колонках пустыми строками
    df_movies_content['overview'] = df_movies_content['overview'].fillna('')
    df_movies_content['genres_list'] = df_movies_content['genres_list'].fillna('')
    df_movies_content['keywords_list'] = df_movies_content['keywords_list'].fillna('')

    def preprocess_text(text):
        if pd.isna(text) or text == '':
            return ''
        # Удаление специальных символов, приведение к нижнему регистру
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text, re.I|re.A)
        text = text.lower()
        text = text.strip()
        return text

    # Применяем предобработку к текстовым полям
    # (Жанры и ключевые слова уже в нижнем регистре и без спецсимволов из-за STRING_AGG, но overview нужно почистить)
    df_movies_content['overview_cleaned'] = df_movies_content['overview'].apply(preprocess_text)
    
    # Создаем одну большую строку "контента" для каждого фильма
    # Увеличим вес для overview, повторяя его (простой способ), и добавим жанры и ключевые слова
    # Ключевые слова и жанры могут содержать пробелы внутри названий (e.g., "science fiction").
    # Мы их уже объединили через STRING_AGG с пробелами между словами.
    # Для TF-IDF лучше, если "science fiction" будет как "sciencefiction" или два отдельных слова.
    # Пока оставим как есть (пробелы между словами жанров/ключевых слов), TF-IDF это обработает.
    # Можно было бы удалить пробелы из многословных жанров/ключевых слов перед объединением.
    
    df_movies_content['content_soup'] = (
        df_movies_content['overview_cleaned'] + ' ' +                # Обзор
        df_movies_content['overview_cleaned'] + ' ' +                # Обзор (повтор для веса)
        (df_movies_content['genres_list'].str.replace(' ', '') + ' ') * 3 +  # Жанры (удаляем пробелы внутри названий жанров)
        (df_movies_content['keywords_list'].str.replace(' ', '') + ' ') * 2     # Ключевые слова (аналогично)
    )
    
    # Удаляем строки, где content_soup мог остаться пустым (если все поля были NaN)
    df_movies_content = df_movies_content[df_movies_content['content_soup'].str.strip() != '']

    print("\nПервые 5 строк с подготовленным 'content_soup':")
    # Выведем только нужные колонки для просмотра
    print(df_movies_content[['movie_id', 'title', 'content_soup']].head())
    
    # Сохраним movie_id и title для последующего маппинга индексов матрицы схожести
    movie_indices = pd.Series(df_movies_content.index, index=df_movies_content['movie_id'])
    movie_titles = pd.Series(df_movies_content['title'].values, index=df_movies_content['movie_id'])

else:
    print("DataFrame df_movies_content пуст, шаги по подготовке контента пропущены.")


Первые 5 строк с подготовленным 'content_soup':
   movie_id                        title  \
0         1                    Toy Story   
1         2                      Jumanji   
2         3             Grumpier Old Men   
3         4            Waiting to Exhale   
4         5  Father of the Bride Part II   

                                        content_soup  
0  led by woody andys toys live happily in his ro...  
1  when siblings judy and peter discover an encha...  
2  a family wedding reignites the ancient feud be...  
3  cheated on mistreated and stepped on the women...  
4  just when george banks has recovered from his ...  


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

if not df_movies_content.empty and 'content_soup' in df_movies_content.columns:
    print("\nTF-IDF векторизация и расчет матрицы схожести...")
    
    # Инициализация TF-IDF Vectorizer
    # stop_words='english' - удаляем распространенные английские слова
    # ngram_range=(1, 2) - рассматриваем одиночные слова и биграммы (пары слов)
    tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=3, max_df=0.8)
    
    # Применяем TF-IDF к нашему 'content_soup'
    tfidf_matrix = tfidf_vectorizer.fit_transform(df_movies_content['content_soup'])
    
    print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}") # (количество фильмов, количество уникальных слов/биграмм)
    
    # Расчет косинусного сходства между всеми парами фильмов
    # Это может потребовать значительной памяти для большого количества фильмов!
    # Для ~9000 фильмов матрица будет (9000, 9000) * sizeof(float)
    # (~9000*9000*4 байта ~= 300MB), что приемлемо.
    print("Расчет косинусной схожести...")
    cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
    
    print(f"Размер матрицы схожести: {cosine_sim_matrix.shape}")
    print("Матрица схожести успешно рассчитана.")

else:
    print("Нет данных для TF-IDF векторизации.")
    cosine_sim_matrix = None # или пустой массив


TF-IDF векторизация и расчет матрицы схожести...
Размер TF-IDF матрицы: (9081, 17579)
Расчет косинусной схожести...
Размер матрицы схожести: (9081, 9081)
Матрица схожести успешно рассчитана.


In [5]:
def get_content_based_recommendations(movie_id, cosine_sim_matrix, movie_indices_map, movie_titles_map, top_n=10):
    """
    Получает N наиболее похожих фильмов для заданного movie_id.
    
    Параметры:
    - movie_id: ID фильма, для которого ищем рекомендации.
    - cosine_sim_matrix: Предвычисленная матрица косинусной схожести.
    - movie_indices_map: Pandas Series для маппинга movie_id в индекс матрицы.
    - movie_titles_map: Pandas Series для маппинга movie_id в название фильма.
    - top_n: Количество рекомендуемых фильмов.
    
    Возвращает:
    - DataFrame с рекомендованными фильмами (movie_id, title, similarity_score).
    """

    
    if movie_id not in movie_indices_map:
        print(f"Фильм с ID {movie_id} не найден в данных.")
        return pd.DataFrame()

    # Получаем индекс фильма в матрице схожести
    idx = movie_indices_map[movie_id]
    
    # Получаем оценки схожести этого фильма со всеми остальными
    # sim_scores представляет собой список кортежей (индекс_фильма, оценка_схожести)
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    
    # Сортируем фильмы по убыванию оценок схожести
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Выбираем top_n фильмов (исключая сам исходный фильм, который будет первым с оценкой 1.0)
    # sim_scores[0] - это сам фильм, поэтому начинаем с sim_scores[1]
    top_similar_movies_indices = [i[0] for i in sim_scores[1:top_n+1]]
    top_similar_movies_scores = [i[1] for i in sim_scores[1:top_n+1]]
    
    # Получаем ID и названия рекомендованных фильмов
    # Обратный маппинг: из индекса матрицы в movie_id
    recommended_movie_ids = [movie_indices_map.index[movie_indices_map == i][0] for i in top_similar_movies_indices]
    recommended_titles = [movie_titles_map[mid] for mid in recommended_movie_ids]
        
    recommendations_df = pd.DataFrame({
        'movie_id': recommended_movie_ids,
        'title': recommended_titles,
        'similarity_score': top_similar_movies_scores
    })
    
    return recommendations_df

# Пример использования (если все предыдущие ячейки отработали)
if cosine_sim_matrix is not None and not df_movies_content.empty:
    # Выберем случайный movie_id из нашего DataFrame для примера
    # (убедимся, что он есть в movie_indices)
    if not movie_indices.empty:
        example_movie_id =movie_indices.sample(1).index[0] # 3446
        example_movie_title = movie_titles[example_movie_id]
        print(f"\n--- Рекомендации для фильма: '{example_movie_title}' (ID: {example_movie_id}) ---")
        
        recommendations = get_content_based_recommendations(example_movie_id, cosine_sim_matrix, movie_indices, movie_titles)
        if not recommendations.empty:
            print(recommendations)
        else:
            print("Не удалось получить рекомендации.")
    else:
        print("Нет данных для выбора примера фильма.")
else:
    print("\nМатрица схожести не рассчитана, примеры рекомендаций не будут показаны.")


--- Рекомендации для фильма: 'Stagecoach' (ID: 7072) ---
   movie_id                                title  similarity_score
0    107412          Kidnapping, Caucasian Style          0.200289
1      6140                              Tenebre          0.199433
2      6659                              Tremors          0.164348
3      3306                           The Circus          0.161958
4      2026                  Disturbing Behavior          0.151774
5      1962                   Driving Miss Daisy          0.150883
6      6909                              The Eye          0.150767
7      2501                          October Sky          0.149133
8       839             The Crow: City of Angels          0.133235
9     86014  Diary of a Wimpy Kid: Rodrick Rules          0.132507


In [6]:
from surprise import Dataset, Reader
from surprise import KNNBasic, SVD
from surprise.model_selection import cross_validate, train_test_split
from collections import defaultdict # Для функции get_top_n_recommendations
from surprise import accuracy

print("Библиотека surprise импортирована.")

Библиотека surprise импортирована.


In [7]:
print("\nЗагрузка данных для коллаборативной фильтрации...")
query_ratings = "SELECT user_id, movie_id, rating FROM ratings;"

try:
    df_ratings = pd.read_sql_query(query_ratings, engine)
    print(f"Загружено {len(df_ratings)} оценок.")
    
    if df_ratings.empty:
        print("Нет данных об оценках для построения коллаборативной системы.")
    else:
        print("\nПервые 5 строк оценок:")
        print(df_ratings.head())

        # Подготовка данных для Surprise
        # Reader ожидает определенный диапазон оценок (rating_scale)
        # У нас в БД рейтинг от 0.5 до 5.0
        reader = Reader(rating_scale=(0.5, 5.0))
        data_surprise = Dataset.load_from_df(df_ratings[['user_id', 'movie_id', 'rating']], reader)
        
        # Мы можем либо использовать весь датасет для обучения (build_full_trainset)
        # Либо разделить на обучающую и тестовую выборки для оценки
        trainset, testset = train_test_split(data_surprise, test_size=0.2, random_state=42)
        
        # Для генерации рекомендаций всем пользователям обычно обучают на всем датасете:
        full_trainset = data_surprise.build_full_trainset()
        
        print("Данные успешно подготовлены для библиотеки surprise.")

except Exception as e:
    print(f"Ошибка при загрузке или подготовке данных об оценках: {e}")
    data_surprise = None
    full_trainset = None
    testset = None


Загрузка данных для коллаборативной фильтрации...
Загружено 99810 оценок.

Первые 5 строк оценок:
   user_id  movie_id  rating
0        1        31     2.5
1        1      1029     3.0
2        1      1061     3.0
3        1      1129     2.0
4        1      1172     4.0
Данные успешно подготовлены для библиотеки surprise.


In [8]:
models_cf = {} # Словарь для хранения обученных моделей

if full_trainset is not None:
    print("\nОбучение моделей коллаборативной фильтрации...")
    
    # 1. User-Based KNN
    print("Обучение User-Based KNN...")
    sim_options_user = {'name': 'cosine', 'user_based': True}
    knn_user_based = KNNBasic(sim_options=sim_options_user, verbose=False)
    knn_user_based.fit(full_trainset)
    models_cf['KNNUserBased'] = knn_user_based
    print("User-Based KNN обучен.")

    # 2. Item-Based KNN
    print("Обучение Item-Based KNN...")
    sim_options_item = {'name': 'cosine', 'user_based': False} # item_based
    knn_item_based = KNNBasic(sim_options=sim_options_item, verbose=False)
    knn_item_based.fit(full_trainset)
    models_cf['KNNItemBased'] = knn_item_based
    print("Item-Based KNN обучен.")
    
    # 3. SVD
    print("Обучение SVD...")
    svd_model = SVD(n_factors=100, n_epochs=20, random_state=42, verbose=False) # n_factors и n_epochs можно тюнить
    svd_model.fit(full_trainset)
    models_cf['SVD'] = svd_model
    print("SVD обучен.")
    
    # (Опционально) Оценка моделей на тестовой выборке
# (Опционально) Оценка моделей на тестовой выборке
    if testset is not None:
        print("\nОценка моделей на тестовой выборке (RMSE, MAE):")
        for name, model in models_cf.items():
            predictions = model.test(testset)
            # Теперь используем импортированный модуль accuracy
            rmse_val = accuracy.rmse(predictions, verbose=False) 
            mae_val = accuracy.mae(predictions, verbose=False)
            print(f"{name}: RMSE={rmse_val:.4f}, MAE={mae_val:.4f}")
else:
    print("Нет данных для обучения моделей коллаборативной фильтрации.")


Обучение моделей коллаборативной фильтрации...
Обучение User-Based KNN...
User-Based KNN обучен.
Обучение Item-Based KNN...
Item-Based KNN обучен.
Обучение SVD...
SVD обучен.

Оценка моделей на тестовой выборке (RMSE, MAE):
KNNUserBased: RMSE=0.8607, MAE=0.6537
KNNItemBased: RMSE=0.9146, MAE=0.7116
SVD: RMSE=0.6464, MAE=0.5003


In [9]:
def get_top_n_collaborative_recommendations(model, user_id, n=10, trainset=None, all_movie_titles=None):
    """
    Генерирует топ-N рекомендаций для пользователя на основе обученной модели Surprise.
    
    Параметры:
    - model: Обученная модель Surprise.
    - user_id: ID пользователя, для которого генерируются рекомендации.
    - n: Количество рекомендаций.
    - trainset: Полный обучающий набор (full_trainset), нужен для получения списка фильмов, которые пользователь УЖЕ оценил.
    - all_movie_titles: Pandas Series (index=movie_id, values=title) для получения названий фильмов.

    Возвращает:
    - DataFrame с рекомендованными фильмами (movie_id, title, estimated_rating).
    """
    if trainset is None:
        print("Ошибка: trainset не предоставлен.")
        return pd.DataFrame()
        
    # Сначала получаем список всех ID фильмов
    all_movie_inner_ids = list(trainset.all_items()) # Внутренние ID фильмов в Surprise
    
    # Конвертируем ID пользователя во внутренний ID Surprise
    try:
        user_inner_id = trainset.to_inner_uid(user_id)
    except ValueError:
        print(f"Пользователь с ID {user_id} не найден в обучающих данных.")
        return pd.DataFrame()
        
    # Фильмы, которые пользователь уже оценил (их нужно исключить из рекомендаций)
    user_rated_items_inner_ids = {item_inner_id for (item_inner_id, rating) in trainset.ur[user_inner_id]}
    
    # Предсказываем оценки для всех фильмов, которые пользователь НЕ оценил
    predictions = []
    for movie_inner_id in all_movie_inner_ids:
        if movie_inner_id not in user_rated_items_inner_ids:
            # Конвертируем внутренний ID фильма обратно в оригинальный movie_id
            original_movie_id = trainset.to_raw_iid(movie_inner_id)
            predicted_rating = model.predict(user_id, original_movie_id).est
            predictions.append((original_movie_id, predicted_rating))
            
    # Сортируем предсказания по убыванию предполагаемой оценки
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Выбираем топ-N
    top_n_recs = []
    for movie_id_rec, est_rating in predictions[:n]:
        title = all_movie_titles.get(movie_id_rec, "Название не найдено")
        top_n_recs.append({'movie_id': movie_id_rec, 'title': title, 'estimated_rating': est_rating})
        
    return pd.DataFrame(top_n_recs)

# Пример использования
if 'SVD' in models_cf and full_trainset is not None:
    # Нужен список всех названий фильмов (мы его уже создавали для Content-Based)
    # Если нет, загрузим его снова
    if 'movie_titles' not in locals() or movie_titles.empty:
        print("Загрузка названий фильмов для рекомендаций...")
        movies_info_query = "SELECT movie_id, title FROM movies;"
        df_movies_info_for_titles = pd.read_sql_query(movies_info_query, engine)
        movie_titles = pd.Series(df_movies_info_for_titles.title.values, index=df_movies_info_for_titles.movie_id).fillna("Название не найдено")

    # Выберем случайного пользователя для примера
    # (Убедимся, что пользователь есть в df_ratings)
    if not df_ratings.empty:
        example_user_id = df_ratings['user_id'].sample(1).iloc[0]
        print(f"\n--- Коллаборативные рекомендации для пользователя ID: {example_user_id} (используя SVD) ---")
        
        svd_recommendations = get_top_n_collaborative_recommendations(
            models_cf['SVD'], 
            user_id=example_user_id, 
            n=10, 
            trainset=full_trainset,
            all_movie_titles=movie_titles
        )
        if not svd_recommendations.empty:
            print(svd_recommendations)
        else:
            print("Не удалось получить коллаборативные рекомендации.")
    else:
        print("Нет данных о пользователях для примера.")
else:
    print("\nМодели коллаборативной фильтрации не обучены, примеры рекомендаций не будут показаны.")


--- Коллаборативные рекомендации для пользователя ID: 475 (используя SVD) ---
   movie_id                     title  estimated_rating
0       593  The Silence of the Lambs          4.794123
1       913        The Maltese Falcon          4.335816
2       527          Schindler's List          4.294778
3      1276            Cool Hand Luke          4.151103
4       293    Leon: The Professional          4.112907
5       306         Three Colors: Red          4.097865
6       899       Singin' in the Rain          4.097385
7       235                   Ed Wood          4.081687
8       318  The Shawshank Redemption          4.069144
9      1089            Reservoir Dogs          4.061503


In [10]:
def get_hybrid_recommendations(user_id, example_movie_id, 
                               content_sim_matrix, content_movie_indices, content_movie_titles,
                               cf_model, cf_trainset, cf_all_movie_titles,
                               top_n_content=20, top_n_cf=20, final_top_n=10,
                               w_content=0.3, w_collaborative=0.7,
                               rating_min=0.5, rating_max=5.0):
    """
    Генерирует гибридные рекомендации, смешивая Content-Based и Collaborative Filtering.
    """
    print(f"\n--- Генерация гибридных рекомендаций для пользователя ID: {user_id}, "
          f"используя фильм ID: {example_movie_id} как пример для контентной части ---")

    # 1. Получаем Content-Based рекомендации (похожие на example_movie_id)
    content_recs_df = get_content_based_recommendations(
        example_movie_id, 
        content_sim_matrix, 
        content_movie_indices, 
        content_movie_titles, 
        top_n=top_n_content
    )
    if content_recs_df.empty:
        print("Не удалось получить Content-Based рекомендации.")
        # Можно вернуть только CF или пустой DataFrame
    
    # Нормализуем similarity_score (он уже от 0 до 1, но на всякий случай убедимся)
    if not content_recs_df.empty:
        content_recs_df['norm_score'] = content_recs_df['similarity_score'] 

    # 2. Получаем Collaborative Filtering (SVD) рекомендации для user_id
    # Важно: get_top_n_collaborative_recommendations уже возвращает фильмы, которые пользователь НЕ оценил
    cf_recs_df = get_top_n_collaborative_recommendations(
        cf_model, 
        user_id, 
        n=top_n_cf, 
        trainset=cf_trainset,
        all_movie_titles=cf_all_movie_titles
    )
    if cf_recs_df.empty:
        print("Не удалось получить Collaborative Filtering рекомендации.")
        # Можно вернуть только Content-Based или пустой DataFrame

    # Нормализуем estimated_rating к диапазону [0, 1]
    if not cf_recs_df.empty:
        cf_recs_df['norm_score'] = (cf_recs_df['estimated_rating'] - rating_min) / (rating_max - rating_min)
        # Ограничим значения, чтобы точно быть в [0, 1]
        cf_recs_df['norm_score'] = cf_recs_df['norm_score'].clip(0, 1)


    # 3. Объединяем и считаем гибридный скор
    hybrid_recs = {} # Словарь: movie_id -> {'title': ..., 'hybrid_score': ...}

    # Добавляем контентные рекомендации
    if not content_recs_df.empty:
        for _, row in content_recs_df.iterrows():
            mid = row['movie_id']
            if mid not in hybrid_recs:
                hybrid_recs[mid] = {'title': row['title'], 'content_score': 0.0, 'cf_score': 0.0}
            hybrid_recs[mid]['content_score'] = row['norm_score']

    # Добавляем коллаборативные рекомендации
    if not cf_recs_df.empty:
        for _, row in cf_recs_df.iterrows():
            mid = row['movie_id']
            if mid not in hybrid_recs:
                hybrid_recs[mid] = {'title': row['title'], 'content_score': 0.0, 'cf_score': 0.0}
            hybrid_recs[mid]['cf_score'] = row['norm_score']
            # Если фильма не было в контентных, возьмем title из CF
            if 'title' not in hybrid_recs[mid] or hybrid_recs[mid]['title'] == "Название не найдено":
                 hybrid_recs[mid]['title'] = row['title']


    # Рассчитываем гибридный скор
    final_recs_list = []
    for mid, scores_data in hybrid_recs.items():
        hybrid_score = (w_content * scores_data['content_score'] + 
                        w_collaborative * scores_data['cf_score'])
        final_recs_list.append({
            'movie_id': mid,
            'title': scores_data['title'],
            'hybrid_score': hybrid_score,
            # 'debug_content_score': scores_data['content_score'], # Для отладки
            # 'debug_cf_score': scores_data['cf_score']          # Для отладки
        })
        
    if not final_recs_list:
        print("Нет данных для формирования гибридных рекомендаций.")
        return pd.DataFrame()

    # 4. Сортируем и берем топ-N
    hybrid_df = pd.DataFrame(final_recs_list)
    hybrid_df.sort_values(by='hybrid_score', ascending=False, inplace=True)
    
    # Исключаем сам example_movie_id из гибридных рекомендаций, если он там есть
    hybrid_df = hybrid_df[hybrid_df['movie_id'] != example_movie_id]
    
    return hybrid_df.head(final_top_n)


# Пример использования гибридной системы
# Убедимся, что все необходимые переменные из предыдущих шагов существуют
if (cosine_sim_matrix is not None and 
    'movie_indices' in locals() and not movie_indices.empty and
    'movie_titles' in locals() and not movie_titles.empty and
    'SVD' in models_cf and models_cf['SVD'] is not None and
    full_trainset is not None and
    not df_ratings.empty):

    # Возьмем того же пользователя и фильм, для которых у нас были примеры
    example_user_id_hybrid = df_ratings['user_id'].sample(1).iloc[0]
    
    # Возьмем фильм, который этот пользователь высоко оценил, как пример для Content-Based
    # Или просто случайный фильм
    user_high_ratings = df_ratings[(df_ratings['user_id'] == example_user_id_hybrid) & (df_ratings['rating'] >= 4.0)]
    if not user_high_ratings.empty:
        example_movie_id_hybrid = user_high_ratings['movie_id'].sample(1).iloc[0]
        example_movie_title_hybrid = movie_titles.get(example_movie_id_hybrid, "Неизвестный фильм")
    else: # Если у пользователя нет высоких оценок, возьмем случайный фильм из тех, для которых есть контент
        example_movie_id_hybrid = movie_indices.sample(1).index[0]
        example_movie_title_hybrid = movie_titles.get(example_movie_id_hybrid, "Неизвестный фильм")

    print(f"\n--- Пример гибридных рекомендаций для пользователя ID: {example_user_id_hybrid} ---")
    print(f"--- (Фильм-пример для контентной части: '{example_movie_title_hybrid}' (ID: {example_movie_id_hybrid})) ---")

    hybrid_recommendations = get_hybrid_recommendations(
        user_id=example_user_id_hybrid,
        example_movie_id=example_movie_id_hybrid,
        content_sim_matrix=cosine_sim_matrix,
        content_movie_indices=movie_indices, # Это pd.Series(df_movies_content.index, index=df_movies_content['movie_id'])
        content_movie_titles=movie_titles,   # Это pd.Series(df_movies_content['title'].values, index=df_movies_content['movie_id'])
        cf_model=models_cf['SVD'],
        cf_trainset=full_trainset,
        cf_all_movie_titles=movie_titles, # Используем тот же маппинг названий
        w_content=0.4, # Дадим чуть больше веса контенту для разнообразия
        w_collaborative=0.6 
    )

    if not hybrid_recommendations.empty:
        print(hybrid_recommendations[['movie_id', 'title', 'hybrid_score']]) # Уберем debug колонки для чистоты
    else:
        print("Не удалось сгенерировать гибридные рекомендации.")
else:
    print("\nНе все компоненты готовы для гибридной системы. Проверьте предыдущие шаги.")


--- Пример гибридных рекомендаций для пользователя ID: 344 ---
--- (Фильм-пример для контентной части: 'Antonia's Line' (ID: 82)) ---

--- Генерация гибридных рекомендаций для пользователя ID: 344, используя фильм ID: 82 как пример для контентной части ---
    movie_id                 title  hybrid_score
20       858         The Godfather      0.540930
21      2467  The Name of the Rose      0.532606
22      1172       Cinema Paradiso      0.531389
23      2762       The Sixth Sense      0.528449
24       926         All About Eve      0.523859
25      3462          Modern Times      0.522634
26       608                 Fargo      0.518974
27      2300         The Producers      0.515996
28      2571            The Matrix      0.515826
29     31658  Howl's Moving Castle      0.514743


In [11]:
import pickle
import numpy as np
import os # Убедись, что os импортирован

# --- Сохранение артефактов ---
artifacts_dir = "streamlit_app_artifacts" # Название папки
os.makedirs(artifacts_dir, exist_ok=True) # Создаем папку, если ее нет

# 1. Для Content-Based
if 'cosine_sim_matrix' in locals() and cosine_sim_matrix is not None:
    np.save(os.path.join(artifacts_dir, 'cosine_similarity_matrix.npy'), cosine_sim_matrix)
    print("Матрица схожести (cosine_similarity_matrix.npy) сохранена.")
else:
    print("Переменная 'cosine_sim_matrix' не найдена или None. Файл НЕ СОХРАНЕН.")

if 'movie_indices' in locals() and not movie_indices.empty:
    with open(os.path.join(artifacts_dir, 'movie_indices.pkl'), 'wb') as f:
        pickle.dump(movie_indices, f)
    print("Маппинг movie_id -> index (movie_indices.pkl) сохранен.")
else:
    print("Переменная 'movie_indices' не найдена или пуста. Файл НЕ СОХРАНЕН.")

if 'movie_titles' in locals() and not movie_titles.empty:
    with open(os.path.join(artifacts_dir, 'movie_titles.pkl'), 'wb') as f:
        pickle.dump(movie_titles, f)
    print("Маппинг movie_id -> title (movie_titles.pkl) сохранен.")
else:
    print("Переменная 'movie_titles' не найдена или пуста. Файл НЕ СОХРАНЕН.")

if 'df_movies_content' in locals() and not df_movies_content.empty:
    # Убедимся, что все нужные колонки есть, прежде чем пытаться их выбрать
    cols_to_save = ['movie_id', 'title', 'overview', 'genres_list', 'poster_path']
    existing_cols = [col for col in cols_to_save if col in df_movies_content.columns]
    if set(cols_to_save) <= set(df_movies_content.columns): # Проверяем, что все нужные колонки есть
        df_movies_display_info = df_movies_content[cols_to_save].copy()
        df_movies_display_info.to_pickle(os.path.join(artifacts_dir, 'movies_display_info.pkl'))
        print("DataFrame с информацией о фильмах (movies_display_info.pkl) сохранен.")
    else:
        missing_display_cols = list(set(cols_to_save) - set(df_movies_content.columns))
        print(f"Не удалось сохранить 'movies_display_info.pkl', отсутствуют колонки: {missing_display_cols} в df_movies_content.")
else:
    print("Переменная 'df_movies_content' не найдена или пуста. Файл 'movies_display_info.pkl' НЕ СОХРАНЕН.")


# 2. Для Collaborative Filtering (SVD)
if 'models_cf' in locals() and 'SVD' in models_cf and models_cf['SVD'] is not None:
    with open(os.path.join(artifacts_dir, 'svd_model.pkl'), 'wb') as f:
        pickle.dump(models_cf['SVD'], f)
    print("Модель SVD (svd_model.pkl) сохранена.")
else:
    print("Модель SVD не найдена или None. Файл НЕ СОХРАНЕН.")

if 'full_trainset' in locals() and full_trainset is not None:
    with open(os.path.join(artifacts_dir, 'surprise_full_trainset.pkl'), 'wb') as f:
        pickle.dump(full_trainset, f)
    print("Полный обучающий набор Surprise (surprise_full_trainset.pkl) сохранен.")
else:
    print("Переменная 'full_trainset' не найдена или None. Файл НЕ СОХРАНЕН.")
    
if 'df_ratings' in locals() and not df_ratings.empty:
    df_ratings.to_pickle(os.path.join(artifacts_dir, 'ratings_data.pkl'))
    print("DataFrame с оценками (ratings_data.pkl) сохранен.")
else:
    print("Переменная 'df_ratings' не найдена или пуста. Файл 'ratings_data.pkl' НЕ СОХРАНЕН.")

# 3. Для User Cold Start (статистика фильмов)
if 'movie_stats' in locals() and not movie_stats.empty: # movie_stats рассчитывается в Ячейке 14
    movie_stats.to_pickle(os.path.join(artifacts_dir, 'movie_stats.pkl'))
    print("DataFrame со статистикой фильмов (movie_stats.pkl) сохранен.")
else:
    print("Переменная 'movie_stats' не найдена или пуста. Файл 'movie_stats.pkl' НЕ СОХРАНЕН.")


print(f"\nПроцесс сохранения артефактов завершен. Проверьте папку: {artifacts_dir}")

Матрица схожести (cosine_similarity_matrix.npy) сохранена.
Маппинг movie_id -> index (movie_indices.pkl) сохранен.
Маппинг movie_id -> title (movie_titles.pkl) сохранен.
DataFrame с информацией о фильмах (movies_display_info.pkl) сохранен.
Модель SVD (svd_model.pkl) сохранена.
Полный обучающий набор Surprise (surprise_full_trainset.pkl) сохранен.
DataFrame с оценками (ratings_data.pkl) сохранен.
Переменная 'movie_stats' не найдена или пуста. Файл 'movie_stats.pkl' НЕ СОХРАНЕН.

Процесс сохранения артефактов завершен. Проверьте папку: streamlit_app_artifacts


In [12]:
# Убедимся, что df_ratings и df_movies_content загружены и содержат нужные данные
if 'df_ratings' not in locals() or df_ratings.empty:
    print("df_ratings не загружен. Запустите ячейку с загрузкой рейтингов.")
    # Можно здесь остановить выполнение ячейки или присвоить None переменным
    # exit() # или return, если это функция
if 'df_movies_content' not in locals() or df_movies_content.empty:
    print("df_movies_content не загружен. Запустите ячейку с его формированием.")
    # exit()

# Переменные для хранения найденных ID
newbie_user_id = None
genre_enthusiast_user_id = None
explorer_user_id = None
target_genre_display = "Science Fiction" # Переменная для отображения

# --- 1. "Новичок" (пользователь с наименьшим количеством оценок) ---
if 'df_ratings' in locals() and not df_ratings.empty:
    user_rating_counts = df_ratings['user_id'].value_counts()
    if not user_rating_counts.empty:
        newbie_user_id = user_rating_counts.idxmin()
        newbie_ratings_count = user_rating_counts.min()
        print(f"--- 'Новичок' (пример) ---")
        print(f"User ID: {newbie_user_id}, Количество оценок: {newbie_ratings_count}")

        # Посмотрим несколько его оценок, если movie_titles существует
        if 'movie_titles' in locals() and not movie_titles.empty:
            print("Примеры оценок 'Новичка':")
            newbie_merged_ratings = df_ratings[df_ratings['user_id'] == newbie_user_id].merge(
                movie_titles.rename('title'), left_on='movie_id', right_index=True, how='left'
            )
            print(newbie_merged_ratings[['movie_id', 'title', 'rating']].sort_values(by='rating', ascending=False).head())
    else:
        print("Не удалось посчитать количество оценок для пользователей.")
else:
    print("DataFrame df_ratings пуст или не определен. Невозможно найти 'Новичка'.")


# --- 2. "Жанровый Энтузиаст" (например, фанат "Science Fiction") ---
if ('df_ratings' in locals() and not df_ratings.empty and
    'df_movies_content' in locals() and not df_movies_content.empty):
    
    # Убедимся, что movie_id в обоих DataFrame одного типа для корректного merge
    df_ratings['movie_id'] = df_ratings['movie_id'].astype(int)
    df_movies_content['movie_id'] = df_movies_content['movie_id'].astype(int)
    
    ratings_with_genres_df = df_ratings.merge(
        df_movies_content[['movie_id', 'title', 'genres_list']], # title добавляем для отладки/вывода
        on='movie_id',
        how='left'
    )
    ratings_with_genres_df['genres_list'] = ratings_with_genres_df['genres_list'].fillna('')

    target_genre_search_term = "science fiction" # Для поиска в genres_list (в нижнем регистре)
    min_high_ratings_for_genre = 5 # Минимальное количество высоких оценок для этого жанра

    # Фильтруем пользователей, которые ставили высокие оценки фильмам целевого жанра
    genre_specific_high_ratings = ratings_with_genres_df[
        (ratings_with_genres_df['genres_list'].str.lower().str.contains(target_genre_search_term, regex=False, na=False)) &
        (ratings_with_genres_df['rating'] >= 4.0)
    ]
    
    user_genre_rating_counts = genre_specific_high_ratings['user_id'].value_counts()
    
    # Отбираем тех, у кого количество таких оценок >= min_high_ratings_for_genre
    potential_enthusiasts = user_genre_rating_counts[user_genre_rating_counts >= min_high_ratings_for_genre]
    
    if not potential_enthusiasts.empty:
        genre_enthusiast_user_id = potential_enthusiasts.idxmax() # Берем того, у кого больше всего таких оценок
        genre_enthusiast_actual_ratings_count = potential_enthusiasts.max()
        print(f"\n--- 'Жанровый Энтузиаст' (фанат '{target_genre_display}') ---")
        print(f"User ID: {genre_enthusiast_user_id}, Количество высоких оценок (>=4.0) для жанра: {genre_enthusiast_actual_ratings_count}")

        # Посмотрим его Sci-Fi оценки, если movie_titles существует
        if 'movie_titles' in locals() and not movie_titles.empty:
             print(f"Примеры его высоких оценок для '{target_genre_display}':")
             enthusiast_merged_ratings = ratings_with_genres_df[
                 (ratings_with_genres_df['user_id'] == genre_enthusiast_user_id) &
                 (ratings_with_genres_df['genres_list'].str.lower().str.contains(target_genre_search_term, regex=False, na=False)) &
                 (ratings_with_genres_df['rating'] >= 4.0)
             ]
             print(enthusiast_merged_ratings[['movie_id', 'title', 'rating']].sort_values(by='rating', ascending=False).head())
    else:
        print(f"\nНе найдено явных 'Жанровых Энтузиастов' для '{target_genre_display}' (с >= {min_high_ratings_for_genre} оценками >= 4.0).")
        print("Возможные причины: мало данных, слишком строгие критерии или данный жанр не популярен у пользователей с большим числом оценок.")
else:
    print("Необходимые DataFrames (df_ratings, df_movies_content) не загружены. Невозможно найти 'Жанрового Энтузиаста'.")


# --- 3. "Киноман / Эксплорер" (пользователь с большим количеством оценок) ---
if 'user_rating_counts' in locals() and not user_rating_counts.empty: # user_rating_counts из блока "Новичок"
    explorer_user_id = user_rating_counts.idxmax()
    explorer_ratings_count = user_rating_counts.max()
    print(f"\n--- 'Киноман / Эксплорер' (пример) ---")
    print(f"User ID: {explorer_user_id}, Количество оценок: {explorer_ratings_count}")
else:
    # Эта ветка сработает, если user_rating_counts не был создан (например, df_ratings пуст)
    print("Невозможно найти 'Киномана / Эксплорера', так как user_rating_counts не определен.")


# --- Итоговый словарь с User ID для сценариев ---
# (используем .get() с default=None на случай, если какая-то переменная не определилась)
user_scenarios_for_streamlit_and_testing = {
    "Новичок": newbie_user_id,
    f"Жанровый Энтузиаст ({target_genre_display})": genre_enthusiast_user_id,
    "Киноман / Эксплорер": explorer_user_id
}

print("\nПримерные ID пользователей для демонстрации сценариев (могут быть None, если не найдены):")
print(user_scenarios_for_streamlit_and_testing)

--- 'Новичок' (пример) ---
User ID: 668, Количество оценок: 19
Примеры оценок 'Новичка':
       movie_id                     title  rating
99608       296              Pulp Fiction     5.0
99610       593  The Silence of the Lambs     5.0
99611       608                     Fargo     5.0
99621      2997      Being John Malkovich     5.0
99613      1213                GoodFellas     5.0

--- 'Жанровый Энтузиаст' (фанат 'Science Fiction') ---
User ID: 564, Количество высоких оценок (>=4.0) для жанра: 142
Примеры его высоких оценок для 'Science Fiction':
       movie_id                           title  rating
83785      2288                       The Thing     5.0
83478      1655                        Phantoms     5.0
83096       849                Escape from L.A.     5.0
84360      3576                      The Hidden     5.0
83988      2664  Invasion of the Body Snatchers     5.0

--- 'Киноман / Эксплорер' (пример) ---
User ID: 547, Количество оценок: 2386

Примерные ID пользователей 

In [13]:
# ID пользователей из предыдущей ячейки (если они не None)
user_ids_to_test = {
    "Новичок": user_scenarios_for_streamlit_and_testing.get("Новичок"),
    "Жанровый Энтузиаст (Sci-Fi)": user_scenarios_for_streamlit_and_testing.get(f"Жанровый Энтузиаст ({target_genre_display})"),
    "Киноман / Эксплорер": user_scenarios_for_streamlit_and_testing.get("Киноман / Эксплорер")
}

# Убедимся, что все необходимые переменные для функций рекомендаций существуют
# (cosine_sim_matrix, movie_indices, movie_titles, models_cf['SVD'], full_trainset)
if (cosine_sim_matrix is None or 
    'movie_indices' not in locals() or movie_indices.empty or
    'movie_titles' not in locals() or movie_titles.empty or
    'SVD' not in models_cf or models_cf['SVD'] is None or
    full_trainset is None):
    print("ОШИБКА: Не все необходимые компоненты для генерации рекомендаций готовы. "
          "Пожалуйста, убедитесь, что ячейки с Content-Based и Collaborative Filtering были успешно выполнены.")
else:
    for user_type, user_id in user_ids_to_test.items():
        if user_id is None:
            print(f"\n{'='*20} СЦЕНАРИЙ ДЛЯ '{user_type}': Пользователь не найден, тестирование пропускается. {'='*20}")
            continue

        print(f"\n\n{'='*30} ТЕСТИРОВАНИЕ ДЛЯ '{user_type.upper()}' (USER ID: {user_id}) {'='*30}")

        # --- Определяем фильм-пример для Content-Based и Hybrid ---
        # Логика выбора фильма-примера:
        # 1. Новичок: Возьмем фильм, который он оценил высоко (если есть), или популярный фильм.
        # 2. Жанровый Энтузиаст: Возьмем Sci-Fi фильм, который он оценил высоко.
        # 3. Киноман: Возьмем какой-нибудь из его высоко оцененных фильмов, возможно, не самый мейнстримный.
        
        example_movie_id_for_user = None
        example_movie_title_for_user = "Не выбран"

        user_ratings_for_cb = df_ratings[(df_ratings['user_id'] == user_id) & (df_ratings['rating'] >= 4.0)]
        # Фильтруем, чтобы фильм-пример был в нашей контентной базе (movie_indices)
        if not user_ratings_for_cb.empty:
            user_ratings_for_cb = user_ratings_for_cb[user_ratings_for_cb['movie_id'].isin(movie_indices.index)]
        
        if user_type == "Новичок":
            if not user_ratings_for_cb.empty:
                example_movie_id_for_user = user_ratings_for_cb.sort_values(by='rating', ascending=False)['movie_id'].iloc[0]
            else: # Если у новичка нет высоких оценок или они не в контентной базе
                # Возьмем просто очень популярный фильм из нашей контентной базы
                # (например, фильм с наибольшим количеством оценок в df_ratings, который есть в movie_indices)
                popular_movies_in_content = df_ratings['movie_id'].value_counts().loc[lambda x: x.index.isin(movie_indices.index)]
                if not popular_movies_in_content.empty:
                     example_movie_id_for_user = popular_movies_in_content.idxmax()

        elif user_type == f"Жанровый Энтузиаст ({target_genre_display})":
            # Ищем высокооцененный Sci-Fi фильм этого пользователя
            # (ratings_with_genres_df из предыдущей ячейки)
            if 'ratings_with_genres_df' in locals() and not ratings_with_genres_df.empty:
                enthusiast_sf_movies = ratings_with_genres_df[
                    (ratings_with_genres_df['user_id'] == user_id) &
                    (ratings_with_genres_df['genres_list'].str.lower().str.contains(target_genre_search_term, regex=False, na=False)) &
                    (ratings_with_genres_df['rating'] >= 4.5) & # Более строгий порог для примера
                    (ratings_with_genres_df['movie_id'].isin(movie_indices.index)) 
                ]['movie_id']
                if not enthusiast_sf_movies.empty:
                    example_movie_id_for_user = enthusiast_sf_movies.sample(1).iloc[0]
                else: # Если не нашли с оценкой >= 4.5, ищем с >= 4.0
                    enthusiast_sf_movies_lower_threshold = ratings_with_genres_df[
                        (ratings_with_genres_df['user_id'] == user_id) &
                        (ratings_with_genres_df['genres_list'].str.lower().str.contains(target_genre_search_term, regex=False, na=False)) &
                        (ratings_with_genres_df['rating'] >= 4.0) &
                        (ratings_with_genres_df['movie_id'].isin(movie_indices.index)) 
                    ]['movie_id']
                    if not enthusiast_sf_movies_lower_threshold.empty:
                        example_movie_id_for_user = enthusiast_sf_movies_lower_threshold.sample(1).iloc[0]
            # Если все равно не нашли, можно взять известный Sci-Fi фильм из movie_indices
            if example_movie_id_for_user is None:
                 candidate_sf = df_movies_content[
                     df_movies_content['genres_list'].str.lower().str.contains(target_genre_search_term.replace(" ", ""), na=False) & # Ищем "sciencefiction"
                     df_movies_content['movie_id'].isin(movie_indices.index)
                 ]
                 if not candidate_sf.empty:
                      example_movie_id_for_user = candidate_sf['movie_id'].sample(1).iloc[0]


        elif user_type == "Киноман / Эксплорер":
            if not user_ratings_for_cb.empty:
                # Возьмем фильм с высокой оценкой, но не самый очевидный (например, не тот, у которого больше всего оценок)
                # Для примера, отсортируем по movie_id и возьмем какой-то из середины или конца списка
                example_movie_id_for_user = user_ratings_for_cb.sort_values(by='movie_id')['movie_id'].iloc[len(user_ratings_for_cb)//2]
            else: # Если у киномана нет высоких оценок (маловероятно, но на всякий случай)
                 if not movie_indices.empty:
                    example_movie_id_for_user = movie_indices.sample(1).index[0]
        
        if example_movie_id_for_user is None and not movie_indices.empty: # Запасной вариант, если фильм-пример не выбран
            print("Фильм-пример не был выбран для пользователя, берем случайный.")
            example_movie_id_for_user = movie_indices.sample(1).index[0]
            
        if example_movie_id_for_user is not None:
            example_movie_title_for_user = movie_titles.get(example_movie_id_for_user, "Неизвестный фильм-пример")
            print(f"Фильм-пример для Content-Based и Hybrid: '{example_movie_title_for_user}' (ID: {example_movie_id_for_user})")
        else:
            print("Не удалось выбрать фильм-пример для Content-Based и Hybrid для этого пользователя.")
            # В этом случае Content-Based и Hybrid не будут работать для этого пользователя
            # Можно их пропустить или обработать эту ситуацию в функциях рекомендаций
            continue # Переходим к следующему пользователю

        # --- 1. Content-Based Рекомендации ---
        print(f"\n1. Content-Based Рекомендации (похожие на '{example_movie_title_for_user}'):")
        cb_recs = get_content_based_recommendations(
            example_movie_id_for_user, 
            cosine_sim_matrix, 
            movie_indices, 
            movie_titles
        )
        if not cb_recs.empty:
            print(cb_recs)
        else:
            print("Не удалось получить Content-Based рекомендации.")

        # --- 2. Collaborative Filtering (SVD) Рекомендации ---
        print(f"\n2. Collaborative Filtering (SVD) Рекомендации для User ID: {user_id}:")
        cf_recs = get_top_n_collaborative_recommendations(
            models_cf['SVD'], 
            user_id, 
            trainset=full_trainset,
            all_movie_titles=movie_titles
        )
        if not cf_recs.empty:
            print(cf_recs)
        else:
            print("Не удалось получить Collaborative Filtering рекомендации.")
            
        # --- 3. Hybrid Рекомендации ---
        print(f"\n3. Hybrid Рекомендации для User ID: {user_id} (на основе '{example_movie_title_for_user}'):")
        hybrid_recs = get_hybrid_recommendations(
            user_id=user_id,
            example_movie_id=example_movie_id_for_user,
            content_sim_matrix=cosine_sim_matrix,
            content_movie_indices=movie_indices,
            content_movie_titles=movie_titles,
            cf_model=models_cf['SVD'],
            cf_trainset=full_trainset,
            cf_all_movie_titles=movie_titles,
            w_content=0.4, 
            w_collaborative=0.6 
        )
        if not hybrid_recs.empty:
            print(hybrid_recs[['movie_id', 'title', 'hybrid_score']])
        else:
            print("Не удалось сгенерировать гибридные рекомендации.")



============================== ТЕСТИРОВАНИЕ ДЛЯ 'НОВИЧОК' (USER ID: 668) ==============================
Фильм-пример для Content-Based и Hybrid: 'Pulp Fiction' (ID: 296)

1. Content-Based Рекомендации (похожие на 'Pulp Fiction'):
   movie_id                                     title  similarity_score
0       319                             Shallow Grave          0.210357
1      3120               The Distinguished Gentleman          0.199108
2    113565                             The Sacrament          0.191843
3      3218                                    Poison          0.189362
4     90576                       What's Your Number?          0.187287
5      1153                                  Raw Deal          0.178172
6      7012                               Mr. Destiny          0.176500
7    102252                Legendary Weapons of China          0.170668
8    102880                               After Earth          0.170031
9      8830  Anacondas: The Hunt for the Blood O

In [14]:
# Убедимся, что нужные DataFrames загружены
if ('df_ratings' not in locals() or df_ratings.empty or
    'movie_titles' not in locals() or movie_titles.empty):
    print("ОШИБКА: df_ratings или movie_titles не загружены. Невозможно сгенерировать популярные фильмы.")
else:
    # --- Рассчитываем средний рейтинг и количество оценок для каждого фильма ---
    movie_stats = df_ratings.groupby('movie_id').agg(
        mean_rating=('rating', 'mean'),
        count_ratings=('rating', 'count')
    ).reset_index()
    
    # Добавляем названия фильмов
    movie_stats = movie_stats.merge(movie_titles.rename('title'), left_on='movie_id', right_index=True, how='left')
    movie_stats.dropna(subset=['title'], inplace=True) # Удаляем фильмы без названий, если такие есть

    # --- Вариант 1: Топ-N фильмов по количеству оценок (самые популярные) ---
    # Устанавливаем минимальный порог для количества оценок, чтобы избежать слишком нишевых
    min_ratings_for_popular = 50 # Например, фильм должен иметь хотя бы 50 оценок, чтобы считаться популярным
    
    popular_movies = movie_stats[movie_stats['count_ratings'] >= min_ratings_for_popular].sort_values(
        by='count_ratings', ascending=False
    )
    
    print(f"\n--- Топ-10 самых популярных фильмов (с >= {min_ratings_for_popular} оценками) ---")
    print(popular_movies[['movie_id', 'title', 'count_ratings', 'mean_rating']].head(10))

    # --- Вариант 2: Топ-N фильмов по среднему рейтингу (с учетом порога по количеству оценок) ---
    # Это похоже на "взвешенный рейтинг" или "рейтинг IMDB", но упрощенно.
    # Мы просто отфильтруем по количеству оценок, а потом отсортируем по среднему рейтингу.
    min_ratings_for_high_rated = 100 # Фильм должен иметь хотя бы 100 оценок, чтобы его высокий средний рейтинг был значимым
    
    high_rated_movies = movie_stats[movie_stats['count_ratings'] >= min_ratings_for_high_rated].sort_values(
        by='mean_rating', ascending=False
    )
    
    print(f"\n--- Топ-10 фильмов с самым высоким средним рейтингом (с >= {min_ratings_for_high_rated} оценками) ---")
    print(high_rated_movies[['movie_id', 'title', 'mean_rating', 'count_ratings']].head(10))

    # --- Функция для получения рекомендаций для нового пользователя ---
    def get_new_user_recommendations(movie_stats_df, top_n=10, min_ratings_threshold=100, sort_by='mean_rating'):
        """
        Возвращает топ-N фильмов для нового пользователя.
        Сортирует по 'mean_rating' (по умолчанию) или 'count_ratings'.
        Применяет min_ratings_threshold к количеству оценок.
        """
        if sort_by not in ['mean_rating', 'count_ratings']:
            print("Ошибка: sort_by должен быть 'mean_rating' или 'count_ratings'.")
            return pd.DataFrame()
            
        filtered_movies = movie_stats_df[movie_stats_df['count_ratings'] >= min_ratings_threshold]
        
        if filtered_movies.empty:
            print(f"Предупреждение: Нет фильмов, удовлетворяющих порогу в {min_ratings_threshold} оценок. "
                  "Попробуйте уменьшить порог.")
            # Попробуем с меньшим порогом или без него
            filtered_movies = movie_stats_df.copy()
            if filtered_movies.empty:
                 print("Нет данных о фильмах для рекомендаций.")
                 return pd.DataFrame()


        recommended_movies = filtered_movies.sort_values(by=sort_by, ascending=False).head(top_n)
        
        return recommended_movies[['movie_id', 'title', 'mean_rating', 'count_ratings']]

    # Пример использования функции для нового пользователя
    print("\n--- Пример рекомендаций для нового пользователя (сортировка по среднему рейтингу) ---")
    new_user_recs = get_new_user_recommendations(movie_stats, min_ratings_threshold=100, sort_by='mean_rating')
    print(new_user_recs)
    
    print("\n--- Пример рекомендаций для нового пользователя (сортировка по популярности) ---")
    new_user_recs_popular = get_new_user_recommendations(movie_stats, min_ratings_threshold=50, sort_by='count_ratings')
    print(new_user_recs_popular)


--- Топ-10 самых популярных фильмов (с >= 50 оценками) ---
      movie_id                       title  count_ratings  mean_rating
321        356                Forrest Gump            341     4.054252
266        296                Pulp Fiction            324     4.256173
284        318    The Shawshank Redemption            311     4.487138
525        593    The Silence of the Lambs            304     4.138158
232        260                   Star Wars            291     4.221649
427        480               Jurassic Park            274     3.706204
2058      2571                  The Matrix            259     4.183398
0            1                   Toy Story            247     3.872470
472        527            Schindler's List            244     4.303279
522        589  Terminator 2: Judgment Day            237     4.006329

--- Топ-10 фильмов с самым высоким средним рейтингом (с >= 100 оценками) ---
      movie_id                            title  mean_rating  count_ratings
692  

In [15]:
# Убедимся, что компоненты для Content-Based готовы
if (cosine_sim_matrix is None or 
    'movie_indices' not in locals() or movie_indices.empty or
    'movie_titles' not in locals() or movie_titles.empty):
    print("ОШИБКА: Компоненты для Content-Based рекомендаций не готовы.")
else:
    # Симулируем "новый" фильм. Возьмем какой-нибудь фильм из нашей базы,
    # для которого есть контентные данные (т.е. он есть в movie_indices).
    # Предположим, у него еще нет оценок.
    
    if not movie_indices.empty:
        # Возьмем случайный фильм из тех, для которых есть контент
        simulated_new_movie_id = movie_indices.sample(1).index[0]
        simulated_new_movie_title = movie_titles.get(simulated_new_movie_id, "Неизвестный 'новый' фильм")
        
        print(f"\n--- Демонстрация для Item Cold Start ---")
        print(f"Симулируем, что фильм '{simulated_new_movie_title}' (ID: {simulated_new_movie_id}) является НОВЫМ.")
        print("Используем Content-Based Filtering, чтобы найти похожие на него фильмы:")
        
        # Получаем контентные рекомендации (фильмы, похожие на наш "новый" фильм)
        # Пользователи, которым нравились эти похожие фильмы, могут быть заинтересованы в "новом" фильме.
        item_cold_start_recs = get_content_based_recommendations(
            simulated_new_movie_id,
            cosine_sim_matrix,
            movie_indices,
            movie_titles,
            top_n=10
        )
        
        if not item_cold_start_recs.empty:
            print(item_cold_start_recs)
        else:
            print(f"Не удалось найти похожие фильмы для '{simulated_new_movie_title}'.")
            
        print("\nЛогика: Если новый фильм похож на эти рекомендованные фильмы по содержанию,")
        print("то его можно предлагать пользователям, которым понравились эти рекомендованные фильмы,")
        print("или показывать его рядом с ними как 'похожий новый фильм'.")
    else:
        print("Нет фильмов в 'movie_indices' для симуляции Item Cold Start.")


--- Демонстрация для Item Cold Start ---
Симулируем, что фильм 'Knife in the Water' (ID: 6858) является НОВЫМ.
Используем Content-Based Filtering, чтобы найти похожие на него фильмы:
   movie_id                        title  similarity_score
0       179                     Mad Love          0.200964
1      8575                 Happenstance          0.179858
2     80083     Dragon Ball Z: Dead Zone          0.162303
3      6987  The Cabinet of Dr. Caligari          0.158953
4       472             I'll Do Anything          0.158586
5      4085            Beverly Hills Cop          0.148277
6     84944                        Rango          0.147913
7    152079            London Has Fallen          0.135333
8      5147            Wild Strawberries          0.132837
9     57223           Dragon Wars: D-War          0.130963

Логика: Если новый фильм похож на эти рекомендованные фильмы по содержанию,
то его можно предлагать пользователям, которым понравились эти рекомендованные фильмы,
или